In [1]:
import sys
import os

# This is the correct way
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # Go up to main folder

if project_root not in sys.path:
    sys.path.insert(0, project_root)   # insert at front (highest priority)

print("Added project root:", project_root)

Added project root: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System


In [2]:
from src.predict import FEATURE_COLUMNS,_preprocess_inputs
from src.model_utils import load_model, load_medians, load_fences


In [3]:
model = load_model()
medians = load_medians()
fences = load_fences()

print("✅ Model loaded successfully")
print(f"Feature columns: {FEATURE_COLUMNS}")

2026-08-20 20:52:22,530 INFO agrotree: Model loaded from: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System\models\crop_model.pkl
2026-08-20 20:52:22,533 INFO agrotree: Medians loaded from: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System\models\crop_medians.pkl
2026-08-20 20:52:22,535 INFO agrotree: Fences loaded from: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System\models\iqr_fences.pkl


✅ Model loaded successfully
Feature columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']


In [4]:
def get_decision_path(node, sample, path=None):
    """
    Traverse the tree and return:
    - decision path
    - reached leaf node
    """
    if path is None:
        path = []

    # Reached leaf
    if node.value is not None:
        return path, node

    feature_name = FEATURE_COLUMNS[node.feature]
    threshold = node.threshold
    value = sample[feature_name]

    if value <= threshold:
        path.append(f"{feature_name} ({value}) <= {threshold:.2f}")
        return get_decision_path(node.left, sample, path)
    else:
        path.append(f"{feature_name} ({value}) > {threshold:.2f}")
        return get_decision_path(node.right, sample, path)

In [5]:
sample = {
    "N": 90,
    "P": 50,
    "K": 40,
    "temperature": 25,
    "humidity": 70,
    "ph": 6.5,
    "rainfall": 150
}

# Apply exactly the same preprocessing as prediction
features = _preprocess_inputs(
    sample["N"],
    sample["P"],
    sample["K"],
    sample["temperature"],
    sample["humidity"],
    sample["ph"],
    sample["rainfall"]
)

# Convert back to dictionary for get_decision_path()
processed_sample = dict(zip(FEATURE_COLUMNS, features[0]))

path, leaf = get_decision_path(model.root, processed_sample)

leaf_samples = sum(leaf.class_distribution.values())
predicted_crop = leaf.value
confidence = leaf.class_distribution[predicted_crop] / leaf_samples * 100

print("Predicted Crop :", predicted_crop)
print(f"Confidence     : {confidence:.2f}%")
print("Leaf Samples   :", leaf_samples)
print("Distribution   :", leaf.class_distribution)

2026-08-20 20:52:22,574 INFO agrotree: Fences loaded from: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System\models\iqr_fences.pkl
2026-08-20 20:52:22,586 INFO agrotree:   IQR clipped values: 0
2026-08-20 20:52:22,588 INFO agrotree: Medians loaded from: c:\Users\Dikshan\Desktop\FinalProject\Crop-Recommendation-System\models\crop_medians.pkl
2026-08-20 20:52:22,596 INFO agrotree:   complete: 1, imputed: 0, dropped: 0
2026-08-20 20:52:22,599 INFO agrotree:   log1p applied to K and rainfall


Predicted Crop : jute
Confidence     : 96.99%
Leaf Samples   : 299
Distribution   : {'coconut': 1, 'coffee': 3, 'jute': 290, 'papaya': 1, 'rice': 4}
